# Attention From Raw Tensors

Self-attention is the mechanism that lets each token build a context-dependent representation from earlier positions. The core operation is just matrix multiplication plus masking, but the shapes must be explicit to stay understandable.


## Problem: What Must This Component Compute?

At this point in the model, the input has shape `(B, T, C)`: batch size, sequence length, and channel dimension. The component must transform that tensor without leaking information from future token positions.

We will first write the operation directly with tensors so every dimension is visible. Only after the numerical checks pass will we wrap the operation in a reusable module.

Why this matters later: when an architecture paper claims to improve attention, memory, quantization, or world modeling, it is usually changing one of these constraints: what information can move across positions, how much memory is required, what objective is optimized, or which representations are preserved.


## Queries, Keys, Values, And The Score Matrix

For each position, attention forms a query `q_t`, key `k_t`, and value `v_t`. Similarity scores come from dot products:

\[
s_{ij} = \frac{q_i \cdot k_j}{\sqrt{d_k}}.
\]

The `T x T` matrix of scores says how much each position should read from every other position. In causal language modeling, positions are only allowed to read from the past and the present.


In [ ]:
import torch
from llm_from_scratch.functional import causal_mask, scaled_dot_product_attention

T = 6
D = 8
q = torch.randn(1, T, D)
k = torch.randn(1, T, D)
v = torch.randn(1, T, D)
out, weights = scaled_dot_product_attention(q, k, v, causal_mask(T))
out.shape, weights.shape, weights[0, 0]


In [ ]:
import math
from llm_from_scratch.functional import stable_softmax

q_small = torch.tensor([[[1.0, 0.0], [1.0, 1.0], [0.0, 1.0]]])
k_small = torch.tensor([[[1.0, 0.0], [0.5, 1.0], [0.0, 1.0]]])
v_small = torch.tensor([[[2.0, 0.0], [0.0, 2.0], [1.0, 1.0]]])
mask_small = causal_mask(q_small.size(1))

scores = q_small @ k_small.transpose(-2, -1) / math.sqrt(q_small.size(-1))
masked_scores = scores.masked_fill(~mask_small, float("-inf"))
weights_small = stable_softmax(masked_scores, dim=-1)
manual_out = weights_small @ v_small

manual_out, weights_small


In [ ]:
row_sums = weights_small.sum(dim=-1)
future_weights = torch.triu(weights_small[0], diagonal=1)

assert torch.allclose(row_sums, torch.ones_like(row_sums))
assert torch.allclose(future_weights, torch.zeros_like(future_weights), atol=1e-6)

{"row_sums": row_sums, "future_weights": future_weights}


## Scaling, Causal Masking, And The `T^2` Bottleneck

Without the factor `1 / sqrt(d_k)`, score magnitudes grow with the head dimension and softmax becomes too sharp too early. The causal mask then zeroes out future positions by replacing their scores with negative infinity before normalization.

The expensive part is the score matrix itself: each head computes `T^2` pairwise comparisons. That is why long contexts quickly dominate memory and time.


In [ ]:
from torch.nn import functional as F

torch_out = F.scaled_dot_product_attention(
    q.unsqueeze(1),
    k.unsqueeze(1),
    v.unsqueeze(1),
    attn_mask=causal_mask(T),
    dropout_p=0.0,
).squeeze(1)

assert torch.allclose(out, torch_out, atol=1e-5)

lengths = torch.tensor([64, 128, 512, 2048])
score_counts = lengths ** 2
assert score_counts[-1].item() == 4_194_304

{"torch_match": True, "score_counts_per_head": score_counts.tolist()}


## Abstraction Bridge And Modern Motivation

The project module `CausalSelfAttention` wraps the same steps with learned linear projections, head reshaping, and an output projection. Sparse and subquadratic attention variants exist because the exact score matrix above becomes costly when `T` is large, even before the rest of the transformer block runs.


## Exercise Checkpoint 1

1. If `q` and `k` both have shape `(B, H, T, D)`, what is the shape of `q @ k.transpose(-2, -1)`?
2. Why does the causal mask have to be applied before softmax rather than after the probabilities are already normalized?


## Exercise Checkpoint 2

1. Explain why scaling by `sqrt(d_k)` affects optimization even though it does not change which keys are most similar in exact arithmetic.
2. For `T = 2048`, how many attention scores does one head compute, and why does that motivate sparse or chunked alternatives?
